# Notebook 2: Recommendation Queries

This notebook runs four Cypher recommendation queries against the graph loaded in Notebook 1.

## Before running the notebook

Export the following in your shell:

```bash
export NEO4J_URI="neo4j+s://xxxx.databases.neo4j.io"
export NEO4J_USERNAME="your_username_here"
export NEO4J_PASSWORD="your_password_here"
```

## 1. Install dependencies

In [1]:
%pip install kaleido==0.2.1 \
             neo4j==6.2.0 \
             pandas==3.0.5 \
             tabulate==0.10.0 \
             plotly==5.24.1 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio

from neo4j import GraphDatabase
from tabulate import tabulate

pio.renderers.default = "iframe"

## 3. Credentials

In [3]:
NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

print("Credentials set.")

Credentials set.


## 4. Neo4j connection

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Seed customer, product, and category

We fetch the alphabetically first customer for consistency across runs, the most-purchased product to ensure all queries return results, and the category with the most products for the trending query.

In [5]:
with driver.session() as session:

    customer = session.run("""
        MATCH (c:Customer)
        RETURN c.id AS customer_id, c.name AS customer_name
        ORDER BY c.name ASC
        LIMIT 1
    """).single()

    product = session.run("""
        MATCH (p:Product)<-[r:PURCHASED]-()
        RETURN p.id AS product_id, p.name AS product_name, count(r) AS order_count
        ORDER BY order_count DESC
        LIMIT 1
    """).single()

    top_cat = session.run("""
        MATCH (p:Product)-[:BELONGS_TO]->(cat:Category)
        RETURN cat.name AS category, count(p) AS total
        ORDER BY total DESC
        LIMIT 1
    """).single()

customer_id   = customer["customer_id"]
customer_name = customer["customer_name"]
product_id    = product["product_id"]
product_name  = product["product_name"]
top_category  = top_cat["category"]

print(f"Seed customer : {customer_name}")
print(f"Seed product  : {product_name} ({product['order_count']} orders)")
print(f"Top category  : {top_category} ({top_cat['total']} products)")

Seed customer : Aaron Boyd
Seed product  : Durable Grooming Brush (157 orders)
Top category  : Toys (39 products)


## 6. Query 1: Collaborative filtering

Find customers who bought the same products as the seed customer, then recommend what else those customers purchased.

In [6]:
def collaborative_filtering(tx, customer_id, limit=5):
    result = tx.run("""
        MATCH (target:Customer {id: $customer_id})-[:PURCHASED]->(p:Product)
              <-[:PURCHASED]-(other:Customer)-[:PURCHASED]->(rec:Product)
        WHERE NOT (target)-[:PURCHASED]->(rec)
        RETURN rec.id       AS id,
               rec.name     AS product,
               rec.price    AS price,
               count(other) AS score
        ORDER BY score DESC, id ASC
        LIMIT $limit
    """, customer_id=customer_id, limit=limit)
    return result.data()

with driver.session() as session:
    rows = session.execute_read(collaborative_filtering, customer_id)

print(f"Collaborative filtering recommendations for: {customer_name}\n")
if rows:
    print(tabulate(
        [[r["product"], f"${r['price']:.2f}", r["score"]] for r in rows],
        headers = ["Product", "Price", "Score"]
    ))
    fig = px.bar(
        rows,
        x                   = "score",
        y                   = "product",
        orientation         = "h",
        color_discrete_sequence = ["#1f77b4"],
        title               = f"Collaborative Filtering: Recommendations for {customer_name}",
        labels              = {"score": "Score", "product": "Product"}
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()
    fig.write_image("figure2.png")
else:
    print("No recommendations found for this customer.")

Collaborative filtering recommendations for: Aaron Boyd

Product                             Price      Score
----------------------------------  -------  -------
An Introduction to Public Speaking  $149.80      311
Heavy-Duty Cable Management Box     $424.01      301
Educational Coding Robot            $52.76       294
Minimalist Cable Management Box     $118.94      289
Natural Sunscreen SPF50             $317.95      288


## 7. Query 2: Frequently bought together

Find products that appear in the same order as the seed product.

In [7]:
def frequently_bought_together(tx, product_id, limit=5):
    result = tx.run("""
        MATCH (p:Product {id: $product_id})<-[r1:PURCHASED]-(c:Customer)
              -[r2:PURCHASED]->(other:Product)
        WHERE r1.order_id = r2.order_id
          AND other.id <> $product_id
        RETURN other.id    AS id,
               other.name  AS product,
               other.price AS price,
               count(c)    AS frequency
        ORDER BY frequency DESC, id ASC
        LIMIT $limit
    """, product_id=product_id, limit=limit)
    return result.data()

with driver.session() as session:
    rows = session.execute_read(frequently_bought_together, product_id)

print(f"Frequently bought together with: {product_name}\n")
if rows:
    print(tabulate(
        [[r["product"], f"${r['price']:.2f}", r["frequency"]] for r in rows],
        headers = ["Product", "Price", "Frequency"]
    ))
    fig = px.bar(
        rows,
        x                       = "frequency",
        y                       = "product",
        orientation             = "h",
        color_discrete_sequence = ["#1f77b4"],
        title                   = f"Frequently Bought Together with: {product_name}",
        labels                  = {"frequency": "Frequency", "product": "Product"}
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()
    fig.write_image("figure3.png")
else:
    print("No co-purchase data found for this product.")

Frequently bought together with: Durable Grooming Brush

Product                            Price      Frequency
---------------------------------  -------  -----------
Natural Body Lotion                $43.16             4
Waterproof Hammock                 $82.44             4
Slim-Fit Hiking Vest               $200.41            3
Sugar-Free Collagen Powder         $465.60            3
Adjustable Lumbar Support Cushion  $453.60            3


## 8. Query 3: Content-based filtering

Find products that share the most tags with the seed product.

In [8]:
def content_based(tx, product_id, limit=5):
    result = tx.run("""
        MATCH (p:Product {id: $product_id})-[:TAGGED_WITH]->(t:Tag)
              <-[:TAGGED_WITH]-(rec:Product)
        WHERE rec.id <> $product_id
        RETURN rec.id       AS id,
               rec.name     AS product,
               rec.price    AS price,
               count(t)     AS shared_tags
        ORDER BY shared_tags DESC, id ASC
        LIMIT $limit
    """, product_id=product_id, limit=limit)
    return result.data()

with driver.session() as session:
    rows = session.execute_read(content_based, product_id)

print(f"Content-based recommendations for: {product_name}\n")
if rows:
    print(tabulate(
        [[r["product"], f"${r['price']:.2f}", r["shared_tags"]] for r in rows],
        headers = ["Product", "Price", "Shared Tags"]
    ))
    fig = px.bar(
        rows,
        x                       = "shared_tags",
        y                       = "product",
        orientation             = "h",
        color_discrete_sequence = ["#1f77b4"],
        title                   = f"Content-Based Filtering: Similar Products to {product_name}",
        labels                  = {"shared_tags": "Shared Tags", "product": "Product"}
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()
    fig.write_image("figure4.png")
else:
    print("No content-based recommendations found for this product.")

Content-based recommendations for: Durable Grooming Brush

Product                    Price      Shared Tags
-------------------------  -------  -------------
Durable Dumbbell Pair      $294.14              2
Cold-Pressed Hot Sauce     $234.19              2
Ergonomic Whiteboard       $270.07              2
Waterproof Toiletry Bag    $27.89               2
Smart Mechanical Keyboard  $14.95               2


## 9. Query 4: Trending in category

Find the most purchased products in the top category from 2024-10-01 onwards.

In [9]:
def trending_in_category(tx, category_name, cutoff="2024-10-01", limit=5):
    result = tx.run("""
        MATCH (p:Product)-[:BELONGS_TO]->(cat:Category {name: $category_name})
        MATCH (:Customer)-[r:PURCHASED]->(p)
        WHERE date(r.order_date) >= date($cutoff)
        RETURN p.id     AS id,
               p.name   AS product,
               p.price  AS price,
               count(r) AS purchases
        ORDER BY purchases DESC, id ASC
        LIMIT $limit
    """, category_name=category_name, cutoff=cutoff, limit=limit)
    return result.data()

with driver.session() as session:
    rows = session.execute_read(trending_in_category, top_category)

print(f"Trending products in: {top_category} (from 2024-10-01)\n")
if rows:
    print(tabulate(
        [[r["product"], f"${r['price']:.2f}", r["purchases"]] for r in rows],
        headers = ["Product", "Price", "Purchases"]
    ))
    fig = px.bar(
        rows,
        x                       = "purchases",
        y                       = "product",
        orientation             = "h",
        color_discrete_sequence = ["#1f77b4"],
        title                   = f"Trending in {top_category} (from 2024-10-01)",
        labels                  = {"purchases": "Purchases", "product": "Product"}
    )
    fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.show()
    fig.write_image("figure5.png")
else:
    print("No trending data found for this category and date range.")

Trending products in: Toys (from 2024-10-01)

Product                           Price      Purchases
--------------------------------  -------  -----------
Battery-Free Coding Robot         $82.79            24
Battery-Free Building Blocks Set  $97.92            22
Creative Puzzle Game              $20.78            21
Wooden Magnetic Drawing Board     $108.75           21
Interactive Remote Control Car    $260.36           21


## 10. Teardown

In [10]:
driver.close()
print("Connection closed.")

Connection closed.
